In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_URL = '/content/drive/MyDrive/BigDataIproject/datasets/'

In [ ]:
from operator import concat
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, when, regexp_replace, upper, split, regexp_extract, udf, to_date
from pyspark.sql.types import StringType
import unicodedata

In [ ]:
spark = SparkSession.builder.appName('Project').getOrCreate()

# Process NBAstats

In [ ]:
def load_nbastats(season):
    nbastats_path = f"{data_URL}nbastats/nbastats_{season}.csv"
    nbastats = spark.read.csv(nbastats_path, header=True).cache()

    # Cast selected columns to string
    columns_to_string = [
        'PLAYER1_TEAM_ID', 'PLAYER2_TEAM_ID', 'PLAYER3_TEAM_ID',
        'PLAYER1_ID', 'PLAYER2_ID', 'PLAYER3_ID'
    ]
    for col_name in columns_to_string:
        nbastats = nbastats.withColumn(col_name, col(col_name).cast(StringType()))

    # Filter play events
    nbastats = nbastats.filter(
        (col('EVENTMSGTYPE') <= 5) & (col('EVENTMSGTYPE') != 4)
    )

    # Fill NA for description-related columns
    desc_cols = ['HOMEDESCRIPTION', 'NEUTRALDESCRIPTION', 'VISITORDESCRIPTION']
    for c in desc_cols:
        nbastats = nbastats.withColumn(c, coalesce(col(c), lit("")))

    # Correct: first create DESCRIPTION as empty string (string type)
    nbastats = nbastats.withColumn("DESCRIPTION", lit(""))

    # Combine description fields (string concat)
    for c in desc_cols:
        nbastats = nbastats.withColumn("DESCRIPTION", concat(col("DESCRIPTION"), col(c)))

    # Fix block mask
    mask = col("DESCRIPTION").contains(' BLK)') | col("DESCRIPTION").contains(' BLK')

    nbastats = nbastats \
        .withColumn("PERSON2TYPE", when(mask, col("PERSON3TYPE")).otherwise(col("PERSON2TYPE"))) \
        .withColumn("PERSON2_ID", when(mask, col("PLAYER3_ID")).otherwise(col("PLAYER2_ID"))) \
        .withColumn("PERSON2_NAME", when(mask, col("PLAYER3_NAME")).otherwise(col("PLAYER2_NAME"))) \
        .withColumn("PERSON2_TEAM_ID", when(mask, col("PLAYER3_TEAM_ID")).otherwise(col("PLAYER2_TEAM_ID")))

    # Select final schema
    nbastats = nbastats.select(
        "GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING",
        "DESCRIPTION", "SCOREMARGIN",
        "PLAYER1_ID", "PLAYER1_NAME", "PLAYER1_TEAM_ID",
        "PLAYER1_TEAM_CITY", "PLAYER1_TEAM_NICKNAME",
        "PLAYER1_TEAM_ABBREVIATION",
        "PLAYER2_ID", "PLAYER2_NAME"
    )

    # Split time
    nbastats = nbastats \
        .withColumn("MINUTES", split(col("PCTIMESTRING"), ":").getItem(0).cast("int")) \
        .withColumn("SECONDS", split(col("PCTIMESTRING"), ":").getItem(1).cast("int"))

    # Clean team id
    nbastats = nbastats.withColumn(
        "PLAYER1_TEAM_ID",
        coalesce(col("PLAYER1_TEAM_ID"), lit(""))
    )

    nbastats = nbastats.withColumn(
        "PLAYER2_ID",
        regexp_replace(col("PLAYER2_ID"), r"^0$", "")
    )

    nbastats = nbastats.filter(col("PLAYER1_TEAM_ID") != "")

    # Add season
    nbastats = nbastats.withColumn("SEASON", lit(season))

    return nbastats

# Process Shotdetail

In [ ]:
def load_shotdetail(season):
    shotdetail_path = f"{data_URL}shotdetail/shotdetail_{season}.csv"
    shotdetail = spark.read.csv(shotdetail_path, header=True).cache()
    shotdetail = shotdetail.withColumn("ACTION_TYPE", upper(col("ACTION_TYPE")))
    shotdetail = shotdetail.withColumn(
    "ACTION_TYPE",
    when(
        col("ACTION_TYPE").contains("TURNAROUND BANK"),
        regexp_replace(col("ACTION_TYPE"), "BANK", "JUMP BANK")
    ).otherwise(col("ACTION_TYPE"))
    )
    shotdetail = shotdetail.withColumn(
        "ACTION_TYPE",
        when(
            col("ACTION_TYPE").contains("FADEAWAY SHOT"),
            regexp_replace(col("ACTION_TYPE"), "FADEAWAY", "JUMP FADEAWAY")
        ).otherwise(col("ACTION_TYPE"))
    )
    # Get Any Text before Shot Method appears in String
    pattern = r"^(.*?)(\b\w+ SHOT)$"
    shotdetail = shotdetail.withColumn("SHOT_METHOD", regexp_extract(col("ACTION_TYPE"), pattern, 2)).withColumn("SHOT_ATTRIBUTES", regexp_extract(col("ACTION_TYPE"), pattern, 1))

    shotdetail = shotdetail.select(
    "GAME_ID", "GAME_EVENT_ID", "ACTION_TYPE",
    "SHOT_TYPE", "SHOT_ZONE_BASIC", "SHOT_ZONE_AREA", "SHOT_ZONE_RANGE",
    "SHOT_DISTANCE", "LOC_X", "LOC_Y",
    "SHOT_ATTEMPTED_FLAG", "SHOT_MADE_FLAG",
    "SHOT_ATTRIBUTES", "SHOT_METHOD",
    "GAME_DATE", "HTM", "VTM"
    )

    shotdetail = shotdetail.withColumn("SHOT_ATTRIBUTES", regexp_replace(col("SHOT_ATTRIBUTES"), "-", ""))
    shotdetail = shotdetail.withColumn("SHOT_ZONE_BASIC", regexp_replace(col("SHOT_ZONE_BASIC"), "Left ", ""))
    shotdetail = shotdetail.withColumn("SHOT_ZONE_BASIC", regexp_replace(col("SHOT_ZONE_BASIC"), "Right ", ""))

    shotdetail = shotdetail.withColumn("SHOT_ZONE_AREA", regexp_replace(col("SHOT_ZONE_AREA"), "Left Side Center(LC)", "Left Side"))
    shotdetail = shotdetail.withColumn("SHOT_ZONE_AREA", regexp_replace(col("SHOT_ZONE_AREA"), "Right Side Center(RC)", "Right Side"))
    shotdetail = shotdetail.withColumn("SHOT_ZONE_AREA", regexp_replace(col("SHOT_ZONE_AREA"), "\\(.*?\\)", ''))
    # If regex contains ( ) \ or other special characters, Spark needs double backslashes "\\(", but simple patterns like "^0$" do not.
    shotdetail = shotdetail.withColumn("SHOT_ATTEMPTED_FLAG", col("SHOT_ATTEMPTED_FLAG").cast("boolean"))
    shotdetail = shotdetail.withColumn("SHOT_MADE_FLAG", col("SHOT_MADE_FLAG").cast("boolean"))

    attr_cols = ["ALLEY OOP", "BANK", "CUTTING", "DRIVING", "FLOATING", "PULLUP", "PUTBACK", "RUNNING", "REVERSE", "STEP BACK", "TIP"]
    for attr in attr_cols:
        shotdetail = shotdetail.withColumn(
            attr,
            col("SHOT_ATTRIBUTES").contains(attr)
        )
    shotdetail = shotdetail.drop("SHOT_ATTRIBUTES")
    return shotdetail

# Join Files Together for Both Datasets

In [ ]:
nbastats = (load_nbastats(2024)
            .unionByName(load_nbastats(2023))
            .unionByName(load_nbastats(2022)))

In [ ]:
shotdetail = (load_shotdetail(2024)
              .unionByName(load_shotdetail(2023))
              .unionByName(load_shotdetail(2022)))

In [ ]:
nbastats.show(5)

+--------+--------+------------+------+------------+-----------+-----------+----------+------------------+---------------+-----------------+---------------------+-------------------------+----------+-------------+-------+-------+------+
| GAME_ID|EVENTNUM|EVENTMSGTYPE|PERIOD|PCTIMESTRING|DESCRIPTION|SCOREMARGIN|PLAYER1_ID|      PLAYER1_NAME|PLAYER1_TEAM_ID|PLAYER1_TEAM_CITY|PLAYER1_TEAM_NICKNAME|PLAYER1_TEAM_ABBREVIATION|PLAYER2_ID| PLAYER2_NAME|MINUTES|SECONDS|SEASON|
+--------+--------+------------+------+------------+-----------+-----------+----------+------------------+---------------+-----------------+---------------------+-------------------------+----------+-------------+-------+-------+------+
|22400001|       7|           2|     1|       11:43|       NULL|       NULL|   1642258|Zaccharie Risacher|     1610612737|          Atlanta|                Hawks|                      ATL|          |         NULL|     11|     43|  2024|
|22400001|      10|           2|     1|       11:38|

In [ ]:
shotdetail.show(5)

+--------+-------------+--------------------+--------------+-----------------+-----------------+---------------+-------------+-----+-----+-------------------+--------------+-----------+---------+---+---+---------+-----+-------+-------+--------+------+-------+-------+-------+---------+-----+
| GAME_ID|GAME_EVENT_ID|         ACTION_TYPE|     SHOT_TYPE|  SHOT_ZONE_BASIC|   SHOT_ZONE_AREA|SHOT_ZONE_RANGE|SHOT_DISTANCE|LOC_X|LOC_Y|SHOT_ATTEMPTED_FLAG|SHOT_MADE_FLAG|SHOT_METHOD|GAME_DATE|HTM|VTM|ALLEY OOP| BANK|CUTTING|DRIVING|FLOATING|PULLUP|PUTBACK|RUNNING|REVERSE|STEP BACK|  TIP|
+--------+-------------+--------------------+--------------+-----------------+-----------------+---------------+-------------+-----+-----+-------------------+--------------+-----------+---------+---+---+---------+-----+-------+-------+--------+------+-------+-------+-------+---------+-----+
|22400001|            7|           JUMP SHOT|3PT Field Goal|Above the Break 3| Left Side Center|        24+ ft.|           2

# Making Additional Data Files

## Make Players File

In [ ]:
players1 = nbastats \
  .select("PLAYER1_ID", "PLAYER1_NAME") \
  .dropDuplicates(["PLAYER1_ID", "PLAYER1_NAME"]) \
  .withColumnRenamed("PLAYER1_ID", "ID") \
  .withColumnRenamed("PLAYER1_NAME", "Name")

# PLAYER2 list
players2 = nbastats \
  .select("PLAYER2_ID", "PLAYER2_NAME") \
  .dropDuplicates(["PLAYER2_ID", "PLAYER2_NAME"]) \
  .filter(col("PLAYER2_ID") != "0") \
  .withColumnRenamed("PLAYER2_ID", "ID") \
  .withColumnRenamed("PLAYER2_NAME", "Name")

# UNION both lists
nba_players = players1.unionByName(players2)

# Name ASCII Normalization
# defind udf to normalize ascii
def normalize_ascii_py(s):
    if s is None:
        return None
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')

normalize_ascii = udf(normalize_ascii_py, StringType())

nba_players = nba_players.withColumn("Name", normalize_ascii(col("Name")))

# remove duplicate id
nba_players = nba_players.dropDuplicates(["ID"])
nba_players.write.csv(f"{data_URL}sample/players.csv", header=True, mode="overwrite")

In [ ]:
teams = nbastats \
  .select("PLAYER1_TEAM_ID", "PLAYER1_TEAM_CITY", "PLAYER1_TEAM_NICKNAME", "PLAYER1_TEAM_ABBREVIATION") \
  .withColumnRenamed("PLAYER1_TEAM_ID", "ID") \
  .withColumnRenamed("PLAYER1_TEAM_CITY", "City") \
  .withColumnRenamed("PLAYER1_TEAM_NICKNAME", "Nickname") \
  .withColumnRenamed("PLAYER1_TEAM_ABBREVIATION", "Abbreviation")\
  .filter(col("ID") != "") \
  .dropDuplicates()

teams.write.csv(f"{data_URL}sample/teams.csv", header=True, mode="overwrite")

## Matchups

In [ ]:
games = shotdetail \
  .select("GAME_ID", "GAME_DATE", "HTM", "VTM") \
  .filter(col("GAME_DATE") != "") \
  .dropDuplicates()\
  .withColumn('GAME_DATE', to_date(col('GAME_DATE'), 'yyyyMMdd'))

games = games.join(teams, games.HTM == teams.Abbreviation, how='inner').withColumnRenamed("ID", "HOME_ID").drop(*teams.columns)

games = games.join(teams, games.VTM == teams.Abbreviation, how='inner')\
.withColumnRenamed("ID", "AWAY_ID")\
.drop(*teams.columns)\
.select('GAME_ID', 'GAME_DATE', 'HOME_ID', 'AWAY_ID')

games.write.csv(f"{data_URL}sample/games.csv", header=True, mode="overwrite")

In [ ]:
# Get Opponenet
nbastats = nbastats.join(games, on = 'GAME_ID', how='inner')
nbastats = nbastats.withColumn(
    "OPPONENT",
    when(
        col("PLAYER1_TEAM_ID") == col("HOME_ID"),
        col("AWAY_ID")
    ).otherwise(col("HOME_ID"))
)

In [ ]:
# Drop Uneeded Columns now
nbastats = nbastats.drop('PLAYER1_NAME', 'PLAYER2_NAME', 'PLAYER1_TEAM_CITY', 'PLAYER1_TEAM_NICKNAME', 'PLAYER1_TEAM_ABBREVIATION', 'HOME_ID', 'AWAY_ID')

shotdetail = shotdetail.drop('HTM', 'VTM', 'GAME_DATE')

In [ ]:
# prepare shotdetail
shotdetail_clean = shotdetail.withColumnRenamed("GAME_ID", "SHOT_GAME_ID")

nbadata = nbastats.join(
    shotdetail_clean,
    (nbastats.GAME_ID == shotdetail_clean.SHOT_GAME_ID) &
    (nbastats.EVENTNUM == shotdetail_clean.GAME_EVENT_ID),
    "left"
)

nbadata = nbadata.drop("SHOT_GAME_ID", "GAME_EVENT_ID")

In [ ]:
# Joining with Shotdetail and Create Column for Melt

# nbadata = nbastats.join(
#     shotdetail,
#     on = [nbastats.GAME_ID == shotdetail.GAME_ID, nbastats.EVENTNUM == shotdetail.GAME_EVENT_ID],
#     how='left'
# ).drop(shotdetail["GAME_ID"]).drop('GAME_EVENT_ID')

In [ ]:
nbadata.printSchema()

root
 |-- GAME_ID: string (nullable = true)
 |-- EVENTNUM: string (nullable = true)
 |-- EVENTMSGTYPE: string (nullable = true)
 |-- PERIOD: string (nullable = true)
 |-- PCTIMESTRING: string (nullable = true)
 |-- DESCRIPTION: double (nullable = true)
 |-- SCOREMARGIN: string (nullable = true)
 |-- PLAYER1_ID: string (nullable = true)
 |-- PLAYER1_TEAM_ID: string (nullable = false)
 |-- PLAYER2_ID: string (nullable = true)
 |-- MINUTES: integer (nullable = true)
 |-- SECONDS: integer (nullable = true)
 |-- SEASON: integer (nullable = false)
 |-- GAME_DATE: date (nullable = true)
 |-- OPPONENT: string (nullable = false)
 |-- ACTION_TYPE: string (nullable = true)
 |-- SHOT_TYPE: string (nullable = true)
 |-- SHOT_ZONE_BASIC: string (nullable = true)
 |-- SHOT_ZONE_AREA: string (nullable = true)
 |-- SHOT_ZONE_RANGE: string (nullable = true)
 |-- SHOT_DISTANCE: string (nullable = true)
 |-- LOC_X: string (nullable = true)
 |-- LOC_Y: string (nullable = true)
 |-- SHOT_ATTEMPTED_FLAG: boo

# Write Files to Parquet

In [ ]:
# Player1 data
nbadata.write \
    .mode("overwrite") \
    .partitionBy("PLAYER1_ID") \
    .parquet(f"{data_URL}sample/player_data")

# Player2 data
nbadata.filter(col("PLAYER2_ID") != "") \
    .write \
    .mode("overwrite") \
    .partitionBy("PLAYER2_ID") \
    .parquet(f"{data_URL}sample/player_data")

# Season data
nbadata.write \
    .mode("overwrite") \
    .partitionBy("SEASON") \
    .parquet(f"{data_URL}sample/eason_data")

In [ ]:
# Generate Totals for Each Season (Percentile Plots)
from pyspark.sql import functions as F

# Step 1: Filter
totals = nbadata.filter(F.col("EVENTMSGTYPE") <= 5)

# Step 2: POINTS calculation
totals = totals.withColumn(
    "POINTS",
    F.when(
        (F.col("EVENTMSGTYPE") == 3) & (~F.col("DESCRIPTION").rlike("^MISS")),
        1
    ).otherwise(0)
)

totals = totals.withColumn(
    "POINTS",
    when(
        (F.col("SHOT_TYPE") == "3PT Field Goal") & (F.col("SHOT_MADE_FLAG") == 1),
        3
    ).otherwise(F.col("POINTS"))
)

totals = totals.withColumn(
    "POINTS",
    F.when(
        (F.col("SHOT_TYPE") == "2PT Field Goal") & (F.col("SHOT_MADE_FLAG") == 1),
        2
    ).otherwise(F.col("POINTS"))
)

# -----------------------------
# Step 3: Aggregations
# -----------------------------

# points
points = totals.groupBy("SEASON", "PLAYER1_ID") \
    .agg(F.sum("POINTS").alias("POINTS")) \
    .withColumnRenamed("PLAYER1_ID", "ID")

# FTA
FTA = totals.filter(F.col("EVENTMSGTYPE") == 3) \
    .groupBy("SEASON", "PLAYER1_ID") \
    .agg(F.count("EVENTMSGTYPE").alias("FTA")) \
    .withColumnRenamed("PLAYER1_ID", "ID")

# FGA (mean, count)
FGA = totals.filter(F.col("EVENTMSGTYPE") < 3) \
    .groupBy("SEASON", "PLAYER1_ID") \
    .agg(
        F.mean("SHOT_DISTANCE").alias("SHOT_DISTANCE"),
        F.count("SHOT_DISTANCE").alias("FGA")
    ) \
    .withColumnRenamed("PLAYER1_ID", "ID")

# AST
AST = totals.filter(
    (F.col("EVENTMSGTYPE") < 3)
    & (F.col("SHOT_MADE_FLAG") == 1)
    & (F.col("PLAYER2_ID") != "")
).groupBy("SEASON", "PLAYER2_ID") \
    .agg(F.count("EVENTMSGTYPE").alias("AST")) \
    .withColumnRenamed("PLAYER2_ID", "ID")

# TOV
TOV = totals.filter(F.col("EVENTMSGTYPE") == 5) \
    .groupBy("SEASON", "PLAYER1_ID") \
    .agg(F.count("EVENTMSGTYPE").alias("TOV")) \
    .withColumnRenamed("PLAYER1_ID", "ID")

# -----------------------------
# Step 4: FULL OUTER JOINS
# -----------------------------

totals_df = points \
    .join(FTA, ["SEASON", "ID"], "outer") \
    .join(FGA, ["SEASON", "ID"], "outer") \
    .join(AST, ["SEASON", "ID"], "outer") \
    .join(TOV, ["SEASON", "ID"], "outer")

# replace null with 0
totals_df = totals_df.fillna(0)

# -----------------------------
# Step 5: Advanced Metrics
# -----------------------------

totals_df = totals_df.withColumn(
    "PPP",
    F.col("POINTS") / (F.col("FGA") + 0.44 * F.col("FTA") + F.col("TOV"))
)

totals_df = totals_df.withColumn(
    "TS%",
    (F.col("POINTS") / (2 * (F.col("FGA") + 0.44 * F.col("FTA"))) * 100)
)

totals_df = totals_df.withColumn("AST/TOV", F.col("AST") / F.col("TOV"))
totals_df = totals_df.withColumn("FTr", F.col("FTA") / F.col("FGA"))

# final result
totals_df.show()

+------+-------+------+---+------------------+----+---+---+------------------+------------------+------------------+-------------------+
|SEASON|     ID|POINTS|FTA|     SHOT_DISTANCE| FGA|AST|TOV|               PPP|               TS%|           AST/TOV|                FTr|
+------+-------+------+---+------------------+----+---+---+------------------+------------------+------------------+-------------------+
|  2022| 101108|   686|160|17.494011976047904| 668|524|114|0.8047864852182074| 46.45178764897075|4.5964912280701755|0.23952095808383234|
|  2022|1626145|   743|100|16.138028169014085| 710|417| 74|0.8973429951690821|49.270557029177716| 5.635135135135135|0.14084507042253522|
|  2022|1626149|   250|101|  3.54066985645933| 209| 33| 35|0.8667313826099016| 49.32133838383839|0.9428571428571428|0.48325358851674644|
|  2022|1626153|   317| 60|13.849315068493151| 292|194| 44|0.8747240618101546| 49.78015075376885| 4.409090909090909| 0.2054794520547945|
|  2022|1626156|  1084|216|17.06329113924